In [1]:
import torch
from txtai import Embeddings
import pandas as pd
#from datasets import load_dataset
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


/home/manavka/anaconda3/envs/RAG/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.15.0
True
NVIDIA RTX 2000 Ada Generation Laptop GPU


In [ ]:
train = pd.read_json('pubmedqa/data/pqaa_train_set.json').T
dev = pd.read_json('pubmedqa/data/pqaa_dev_set.json').T
contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
print(train.head())

                                                   QUESTION  \
22117569  Is an advance care planning model feasible in ...   
16097998  Is coeliac disease screening in risk groups ju...   
15223725  Does blood pressure change in treated hyperten...   
17274051  Metastatic carcinoma to the cervical nodes fro...   
27581329  Estimation of basal metabolic rate in Chinese:...   

                                                   CONTEXTS  \
22117569  [An effective advance care planning programme ...   
16097998  [The benefits of serologic screening for coeli...   
15223725  [To determine whether there are differences be...   
17274051  [The aim of the study was to evaluate the outc...   
27581329  [Measurement of basal metabolic rate (BMR) is ...   

                                                     LABELS  \
22117569  [BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...   
16097998                     [BACKGROUND, METHODS, RESULTS]   
15223725                      [OBJECTIVES, METHOD, RE

In [12]:
# dense embeddings
embeddings = Embeddings(path="neuml/pubmedbert-base-embeddings", content=True)
embeddings.index(contexts_train)

Could not reach the Hub ([SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1011)). Using cached commit hash for 'neuml/pubmedbert-base-embeddings'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1793.09it/s]
Could not reach the Hub ([SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1011)). Using cached commit hash for 'neuml/pubmedbert-base-embeddings'.


In [17]:
test_dense = embeddings.search("Is an advance care planning model feasible in community palliative care?")

[d['id'] for d in test_dense]

['0', '866', '50']

<h2>Sparse Embeddings</h2>

In [6]:
train

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...
16097998,Is coeliac disease screening in risk groups ju...,[The benefits of serologic screening for coeli...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Body Mass Index, Bone Density, C...",2005,yes,yes,yes,Long-term dietary compliance in screen-detecte...
15223725,Does blood pressure change in treated hyperten...,[To determine whether there are differences be...,"[OBJECTIVES, METHOD, RESULTS]","[Adult, Aged, Aged, 80 and over, Antihypertens...",2004,maybe,yes,yes,Systolic BP measured by the nurse in treated h...
17274051,Metastatic carcinoma to the cervical nodes fro...,[The aim of the study was to evaluate the outc...,"[BACKGROUND, METHODS, RESULTS]","[Biopsy, Fine-Needle, Carcinoma, Female, Head ...",2007,no,no,no,"In our experience, definitive radiotherapy to ..."
27581329,Estimation of basal metabolic rate in Chinese:...,[Measurement of basal metabolic rate (BMR) is ...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Asian Continental Ancestry Group...",2016,yes,yes,yes,"To date, the newly developed Singapore equatio..."
...,...,...,...,...,...,...,...,...,...
17220021,Is there an increase in the incidence of gbs c...,[Group B Streptococci (GBS) asymptomatically c...,"[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Carrier State, Female, Hum...",2006,no,yes,yes,In the present study we found a significant in...
22075911,Is there a differential in the dental health o...,[Figures from the British Defence Dental Servi...,"[BACKGROUND AND AIM, METHOD, RESULTS]","[Adolescent, Cross-Sectional Studies, DMF Inde...",2011,yes,yes,yes,A significant difference in dental health betw...
10922093,Does open access endoscopy close the door to a...,[The use of open access endoscopy is increasin...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Ambulatory Care, Endoscopy, Gast...",2000,yes,yes,yes,Patients undergoing open access procedures are...
25336163,Are interstitial fluid concentrations of merop...,[To describe the interstitial fluid (ISF) and ...,"[OBJECTIVES, PATIENTS AND METHODS, RESULTS]","[Aged, Anti-Bacterial Agents, Area Under Curve...",2015,no,no,no,This is the first known report of concurrent p...


In [7]:
import bm25s

contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids = train.index.to_numpy()
contexts_train_tokens = bm25s.tokenize(contexts_train, stopwords="en")

sparse_embeddings = bm25s.BM25(corpus=contexts_train)
sparse_embeddings.index(contexts_train_tokens)

assert sparse_embeddings.scores["num_docs"] == len(train)
print(f"Indexed {len(contexts_train):,} contexts with BM25.")

Indexed 1,000 contexts with BM25.


<h2>RRF<h2>

In [ ]:
from typing import List, Dict

def reciprocal_rank_fusion(ranks: List[List[str]], k: int = 60) -> List[str]:
    """
    ranks: list of lists of document IDs from each retrieval method
    k: constant for rank discount
    """
    doc_scores = {doc: 0.0 for doc in set().union(*ranks)}
    for rank_list in ranks:
        for rank, doc in enumerate(rank_list, start=1):
            doc_scores[doc] += 1 / (rank + k)
    return sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)

# Example usage:
list1 = ["docA", "docB", "docC"]
list2 = ["docB", "docC", "docD"]
final = reciprocal_rank_fusion([list1, list2], k=60)
print(final)

def retrieve_and_fuse(query,dense_embeddings, sparse_embeddings):
    dense_list= dense_embeddings.search(query)
    dense_ids = [dict['id'] for dict in dense_list]
    query_tokens = bm25s.tokenize(query, stopwords="en")
    sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=5)
    fused_list = reciprocal_rank_fusion([dense_ids, sparse_ids])
    return fused_list
    

['docB', 'docC', 'docA', 'docD']
